# T17 — Diagnostik GEE: Slope 30 Titik Positif & Ekspor Poligon AOI (Menuju v28)

Notebook ini adalah versi siap pakai di **Google Colab** (cukup klik **Runtime → Run all**) untuk membuka dua bloker utama yang tercatat di laporan T16 dan T17 tanpa melatih ulang model:

1. **Diagnostik Slope 30 Titik (T17):** Mengevaluasi 4 rute proyeksi/grid yang didefinisikan pada `t17_slope_diagnostic.js` di server Google Earth Engine untuk 30 koordinat banjir positif (`slope_srtm >= 5°`).
2. **Ekspor Poligon AOI Independen:** Mengambil geometri resmi `FAO/GAUL/2015/level2` (Kota Semarang + Demak) dan menyimpannya sebagai GeoJSON agar validasi geografis mask pada `v28_delivery` dapat diverifikasi secara independen.

**Catatan:** Seluruh keluaran akan otomatis disimpan ke Google Drive pada folder `Proposal Baru/v28_delivery/diagnostics_t17/`.


In [2]:
# 1. Mount Google Drive & Siapkan Direktori Output v28
import os
import glob
import json
from datetime import datetime, timezone

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MYDRIVE = '/content/drive/MyDrive' if IN_COLAB else os.getcwd()

# Cari folder "Proposal Baru" secara otomatis di Google Drive
proposal_candidates = sorted(glob.glob(os.path.join(MYDRIVE, '**', 'Proposal Baru'), recursive=True))
if proposal_candidates:
    PROPOSAL_DIR = proposal_candidates[0]
else:
    PROPOSAL_DIR = os.path.join(MYDRIVE, '1_Metopen', 'Proposal Baru')
    os.makedirs(PROPOSAL_DIR, exist_ok=True)

OUT_DIR = os.path.join(PROPOSAL_DIR, 'v28_delivery', 'diagnostics_t17')
os.makedirs(OUT_DIR, exist_ok=True)

print('Lingkungan :', 'Colab' if IN_COLAB else 'Lokal')
print('Proposal   :', PROPOSAL_DIR)
print('Output T17 :', OUT_DIR)


Mounted at /content/drive
Lingkungan : Colab
Proposal   : /content/drive/MyDrive/1_Metopen/Proposal Baru
Output T17 : /content/drive/MyDrive/1_Metopen/Proposal Baru/v28_delivery/diagnostics_t17


In [3]:
# 2. Autentikasi Google Earth Engine & Cek Aset
import ee

EE_PROJECT = 'ee-irfan-banjir-2026'

key_candidates = [
    os.path.join(PROPOSAL_DIR, '.credentials', 'gee-service-account.json'),
    os.path.join(PROPOSAL_DIR, '.credentials', 'ee-irfan-banjir-2026-44c092a9f72b.json'),
    os.path.join(MYDRIVE, 'GEE_DEM_Sensitivity_Semarang_Demak', 'gee-service-account.json'),
    os.path.join(MYDRIVE, 'Analisis Flood DEM Susceptibility', 'gee-service-account.json'),
]
key_candidates.extend(sorted(glob.glob(os.path.join(PROPOSAL_DIR, '.credentials', '*.json'))))

initialized = False
used_sa = None
for kp in key_candidates:
    if kp and os.path.isfile(kp):
        try:
            with open(kp, 'r', encoding='utf-8') as fh:
                info = json.load(fh)
            if info.get('type') == 'service_account':
                creds = ee.ServiceAccountCredentials(info['client_email'], kp)
                ee.Initialize(credentials=creds, project=info.get('project_id') or EE_PROJECT)
                initialized = True
                used_sa = info.get('client_email')
                print('GEE terinisialisasi via service account:', used_sa)
                break
        except Exception as exc:
            print('Kandidat key gagal (%s): %s' % (kp, exc))

if not initialized:
    print('Service account tidak ditemukan/gagal. Memakai autentikasi interaktif Colab...')
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print('GEE terinisialisasi via akun interaktif.')

# Verifikasi koneksi & catat metadata aset saat ini
asset_ids = ['FAO/GAUL/2015/level2', 'USGS/SRTMGL1_003', 'COPERNICUS/S1_GRD', 'JRC/GSW1_4/GlobalSurfaceWater']
asset_status = {}
for aid in asset_ids:
    try:
        meta = ee.data.getAsset(aid)
        asset_status[aid] = {
            'status': 'OK',
            'type': meta.get('type'),
            'updateTime': meta.get('updateTime'),
            'version': meta.get('version'),
        }
    except Exception as exc:
        asset_status[aid] = {'status': 'ERROR', 'error': str(exc)[:300]}
    print(f"  {aid}: {asset_status[aid]['status']}")


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


GEE terinisialisasi via service account: nxtirfan@ee-irfan-banjir-2026.iam.gserviceaccount.com


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


  FAO/GAUL/2015/level2: OK
  USGS/SRTMGL1_003: OK
  COPERNICUS/S1_GRD: OK
  JRC/GSW1_4/GlobalSurfaceWater: OK


In [4]:
# 3. Ekstrak & Ekspor Poligon AOI Independen (FAO/GAUL/2015/level2)
gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME', 'Indonesia'))
primary_roi = gaul.filter(ee.Filter.inList('ADM2_NAME', ['Kota Semarang', 'Demak']))
primary_names = primary_roi.aggregate_array('ADM2_NAME').getInfo()

used_fallback = len(primary_names) < 2
selected_roi = gaul.filter(ee.Filter.inList('ADM2_NAME', ['Semarang', 'Demak'])) if used_fallback else primary_roi
selected_roi_names = selected_roi.aggregate_array('ADM2_NAME').getInfo()
roi_selection = 'fallback Semarang + Demak' if used_fallback else 'primary Kota Semarang + Demak'

roi = selected_roi.geometry()
roi_area_km2 = roi.area(maxError=1).divide(1e6).getInfo()
roi_geojson = selected_roi.getInfo()

aoi_geojson_path = os.path.join(OUT_DIR, 'aoi_semarang_demak.geojson')
with open(aoi_geojson_path, 'w', encoding='utf-8') as fh:
    json.dump(roi_geojson, fh, indent=2, ensure_ascii=False)

aoi_meta = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'dataset': 'FAO/GAUL/2015/level2',
    'roi_selection': roi_selection,
    'selected_adm2_names': selected_roi_names,
    'feature_count': len(roi_geojson.get('features', [])),
    'geodesic_area_km2': roi_area_km2,
    'asset_status': asset_status,
}
with open(os.path.join(OUT_DIR, 'aoi_metadata.json'), 'w', encoding='utf-8') as fh:
    json.dump(aoi_meta, fh, indent=2, ensure_ascii=False)

print('ROI terpilih :', roi_selection, selected_roi_names)
print('Luas geodesik:', f'{roi_area_km2:.4f} km2')
print('Disimpan ke  :', aoi_geojson_path)


/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level2! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level2

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level2

  warnings.warn(warning, category=DeprecationWarning)


ROI terpilih : primary Kota Semarang + Demak ['Demak', 'Kota Semarang']
Luas geodesik: 1310.0915 km2
Disimpan ke  : /content/drive/MyDrive/1_Metopen/Proposal Baru/v28_delivery/diagnostics_t17/aoi_semarang_demak.geojson


In [5]:
# 4. Definisi 30 Titik Audit Beku T16 & 4 Rute Diagnostik Slope GEE
AUDIT_POINTS = [
  {
    "audit_id": "training_sample_features.csv:physical_line_128",
    "flood": 1,
    "lon": 110.660809820596,
    "lat": -6.82319865629401,
    "historic_slope_srtm_training_export_deg": 5.10367012023926
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_386",
    "flood": 1,
    "lon": 110.74597010953,
    "lat": -6.85122609315854,
    "historic_slope_srtm_training_export_deg": 5.10363674163818
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_515",
    "flood": 1,
    "lon": 110.543310181433,
    "lat": -6.92776255536552,
    "historic_slope_srtm_training_export_deg": 5.37260675430298
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_975",
    "flood": 1,
    "lon": 110.820620109641,
    "lat": -6.8576939632042,
    "historic_slope_srtm_training_export_deg": 6.92005443572998
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1106",
    "flood": 1,
    "lon": 110.555976426939,
    "lat": -6.93773385501925,
    "historic_slope_srtm_training_export_deg": 6.61613798141479
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1226",
    "flood": 1,
    "lon": 110.541693213921,
    "lat": -6.9080894506433,
    "historic_slope_srtm_training_export_deg": 5.10375928878784
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1342",
    "flood": 1,
    "lon": 110.581578412536,
    "lat": -6.88868584050633,
    "historic_slope_srtm_training_export_deg": 5.5283317565918
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1348",
    "flood": 1,
    "lon": 110.822506571737,
    "lat": -6.85499901735184,
    "historic_slope_srtm_training_export_deg": 5.10361194610596
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1726",
    "flood": 1,
    "lon": 110.824393033834,
    "lat": -6.85499901735184,
    "historic_slope_srtm_training_export_deg": 5.98639488220215
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1860",
    "flood": 1,
    "lon": 110.628470470367,
    "lat": -6.71836526263726,
    "historic_slope_srtm_training_export_deg": 5.12808132171631
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_1935",
    "flood": 1,
    "lon": 110.593166679701,
    "lat": -6.78358295226434,
    "historic_slope_srtm_training_export_deg": 6.07193231582642
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_2176",
    "flood": 1,
    "lon": 110.574841047905,
    "lat": -6.93530840375213,
    "historic_slope_srtm_training_export_deg": 6.01810026168823
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_2509",
    "flood": 1,
    "lon": 110.690993214142,
    "lat": -6.96144937852,
    "historic_slope_srtm_training_export_deg": 5.37252044677734
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_2569",
    "flood": 1,
    "lon": 110.348465596307,
    "lat": -7.0396028082384,
    "historic_slope_srtm_training_export_deg": 10.4980525970459
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_2617",
    "flood": 1,
    "lon": 110.692879676239,
    "lat": -6.96710876480996,
    "historic_slope_srtm_training_export_deg": 5.10368156433105
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_2632",
    "flood": 1,
    "lon": 110.629278954123,
    "lat": -6.72213818683056,
    "historic_slope_srtm_training_export_deg": 6.0180401802063
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3149",
    "flood": 1,
    "lon": 110.824932023004,
    "lat": -6.85446002818136,
    "historic_slope_srtm_training_export_deg": 11.4838371276855
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3269",
    "flood": 1,
    "lon": 110.583195380048,
    "lat": -6.93503890916689,
    "historic_slope_srtm_training_export_deg": 5.75646162033081
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3309",
    "flood": 1,
    "lon": 110.822506571737,
    "lat": -6.85311255525519,
    "historic_slope_srtm_training_export_deg": 7.65811443328857
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3539",
    "flood": 1,
    "lon": 110.59154971219,
    "lat": -6.79705768152613,
    "historic_slope_srtm_training_export_deg": 6.37603616714478
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3576",
    "flood": 1,
    "lon": 110.753246463332,
    "lat": -6.84960912564712,
    "historic_slope_srtm_training_export_deg": 5.37247610092163
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3757",
    "flood": 1,
    "lon": 110.821698087981,
    "lat": -6.85715497403373,
    "historic_slope_srtm_training_export_deg": 5.3724536895752
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3785",
    "flood": 1,
    "lon": 110.587507293412,
    "lat": -6.8043340353275,
    "historic_slope_srtm_training_export_deg": 6.65964412689209
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3797",
    "flood": 1,
    "lon": 110.598287076821,
    "lat": -6.78816436021334,
    "historic_slope_srtm_training_export_deg": 7.1995677947998
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_3924",
    "flood": 1,
    "lon": 110.748126066212,
    "lat": -6.8509565985733,
    "historic_slope_srtm_training_export_deg": 7.19947910308838
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_4164",
    "flood": 1,
    "lon": 110.822506571737,
    "lat": -6.85257356608471,
    "historic_slope_srtm_training_export_deg": 7.62125587463379
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_4858",
    "flood": 1,
    "lon": 110.643831661726,
    "lat": -6.72456363809769,
    "historic_slope_srtm_training_export_deg": 5.10366058349609
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_5134",
    "flood": 1,
    "lon": 110.624428051589,
    "lat": -6.81214937829934,
    "historic_slope_srtm_training_export_deg": 5.10368776321411
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_5507",
    "flood": 1,
    "lon": 110.652185993868,
    "lat": -6.72321616517151,
    "historic_slope_srtm_training_export_deg": 5.10365581512451
  },
  {
    "audit_id": "training_sample_features.csv:physical_line_5627",
    "flood": 1,
    "lon": 110.819272636714,
    "lat": -6.8539210390109,
    "historic_slope_srtm_training_export_deg": 6.37591457366943
  }
]

SENTINEL = -9999
ANALYSIS_CRS = 'EPSG:32749'
ANALYSIS_SCALE_M = 30
FLOOD_SLOPE_LIMIT_DEG = 5

points = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([p['lon'], p['lat']], 'EPSG:4326'), {
        'audit_id': p['audit_id'],
        'flood': p['flood'],
        'input_lon': p['lon'],
        'input_lat': p['lat'],
        'historic_slope_srtm_training_export_deg': p['historic_slope_srtm_training_export_deg'],
    })
    for p in AUDIT_POINTS
])

# Rute 1: Jalur label canon (clipped native SRTM -> Terrain.slope)
srtm_native = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(roi)
label_slope = ee.Terrain.slope(srtm_native).rename('slope')
label_native_proj = label_slope.projection()

# Rute 3: Jalur prediktor canon (reproject elevation ke EPSG:32749 30m -> Terrain.slope)
srtm_feature_elev = srtm_native.reproject(crs=ANALYSIS_CRS, scale=ANALYSIS_SCALE_M).clip(roi)
feature_slope = ee.Terrain.slope(srtm_feature_elev).rename('slope_srtm')

# Rute 2 & 4: Kontrol grid analisis eksplisit
analysis_grid = ee.Image.constant(0).reproject(crs=ANALYSIS_CRS, scale=ANALYSIS_SCALE_M).projection()
label_slope_forced_grid = label_slope.reproject(crs=ANALYSIS_CRS, scale=ANALYSIS_SCALE_M).rename('slope')
feature_final_sampling_slope = feature_slope.reproject(crs=ANALYSIS_CRS, scale=ANALYSIS_SCALE_M).clip(roi)

def projection_metadata(projection):
    info = projection.getInfo()
    return {
        'crs': info.get('crs'),
        'affine_transform': info.get('transform'),
        'transform_wkt': projection.transform().getInfo(),
        'nominal_scale_m': projection.nominalScale().getInfo(),
    }

proj_meta = {
    'labelNative': projection_metadata(label_native_proj),
    'analysisGrid': projection_metadata(analysis_grid),
    'featureElevation': projection_metadata(srtm_feature_elev.projection()),
    'featureSlope': projection_metadata(feature_slope.projection()),
    'featureFinalSampling': projection_metadata(feature_final_sampling_slope.projection()),
}
with open(os.path.join(OUT_DIR, 't17_projection_metadata.json'), 'w', encoding='utf-8') as fh:
    json.dump(proj_meta, fh, indent=2, ensure_ascii=False)

def with_diagnostic_bands(slope_img):
    slope_mask_weight = slope_img.mask().reduce(ee.Reducer.min()).rename('slope_mask_weight')
    lt5 = slope_img.lt(FLOOD_SLOPE_LIMIT_DEG)
    lt5_mask_weight = lt5.mask().reduce(ee.Reducer.min()).rename('lt5_mask_weight')
    return (
        slope_img.rename('slope_deg').unmask(SENTINEL, False)
        .addBands(slope_mask_weight.unmask(0, False))
        .addBands(slope_mask_weight.gt(0).rename('slope_valid').unmask(0, False))
        .addBands(lt5.rename('lt5_value').unmask(SENTINEL, False))
        .addBands(lt5_mask_weight.unmask(0, False))
        .addBands(lt5_mask_weight.gt(0).rename('lt5_valid').unmask(0, False))
    )

def sample_route(route, slope_img, src_meta, eval_meta, sample_proj, sample_meta, route_meaning):
    sampled = with_diagnostic_bands(slope_img).sampleRegions(
        collection=points,
        projection=sample_proj,
        geometries=True,
        tileScale=8,
    )
    def _enrich(f):
        snapped = f.geometry().coordinates()
        return f.set({
            'diagnostic_status': 'CURRENT_GEE_DIAGNOSTIC_NOT_HISTORICAL_REPLAY',
            'route': route,
            'route_meaning': route_meaning,
            'roi_selection': roi_selection,
            'roi_selected_names': '|'.join(selected_roi_names),
            'threshold_deg': FLOOD_SLOPE_LIMIT_DEG,
            'threshold_operator': 'strict_lt',
            'masked_slope_sentinel': SENTINEL,
            'source_crs': src_meta['crs'],
            'source_affine_transform_json': json.dumps(src_meta['affine_transform']),
            'source_transform_wkt': src_meta['transform_wkt'],
            'source_nominal_scale_m': src_meta['nominal_scale_m'],
            'evaluation_crs': eval_meta['crs'],
            'evaluation_affine_transform_json': json.dumps(eval_meta['affine_transform']),
            'evaluation_transform_wkt': eval_meta['transform_wkt'],
            'evaluation_nominal_scale_m': eval_meta['nominal_scale_m'],
            'sampling_crs': sample_meta['crs'],
            'sampling_affine_transform_json': json.dumps(sample_meta['affine_transform']),
            'sampling_transform_wkt': sample_meta['transform_wkt'],
            'sampling_nominal_scale_m': sample_meta['nominal_scale_m'],
            'resampling_override': 'none; Earth Engine default applies',
            'roi_clip_applied': 1,
            'snapped_pixel_center_lon': snapped.get(0),
            'snapped_pixel_center_lat': snapped.get(1),
        })
    return sampled.map(_enrich)

results = (
    sample_route(
        'label_native_projection_control', label_slope, proj_meta['labelNative'],
        proj_meta['labelNative'], label_native_proj, proj_meta['labelNative'],
        'Native-projection SRTM label-slope control with strict lt5 test.'
    )
    .merge(sample_route(
        'label_slope_forced_analysis_grid_control', label_slope_forced_grid,
        proj_meta['labelNative'], proj_meta['analysisGrid'], analysis_grid, proj_meta['analysisGrid'],
        'Explicitly forced-grid label-slope control.'
    ))
    .merge(sample_route(
        'feature_srtm_slope_analysis_grid', feature_slope,
        proj_meta['featureElevation'], proj_meta['featureSlope'], feature_slope.projection(),
        proj_meta['featureSlope'],
        'Feature route: reproject SRTM elevation to EPSG:32749 / 30 m before calculating terrain slope.'
    ))
    .merge(sample_route(
        'feature_slope_final_sampling_grid_control', feature_final_sampling_slope,
        proj_meta['featureElevation'], proj_meta['featureFinalSampling'], analysis_grid, proj_meta['analysisGrid'],
        'Feature slope explicitly evaluated on final analysis grid.'
    ))
)
print('Siap mengeksekusi 4 rute x 30 titik = 120 baris...')


Siap mengeksekusi 4 rute x 30 titik = 120 baris...


In [6]:
# 5. Jalankan Evaluasi GEE & Simpan CSV ke Google Drive
import pandas as pd

fc_info = results.getInfo()
rows = [feat['properties'] for feat in fc_info.get('features', [])]
df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, 't17_slope_30point_diagnostic.csv')
df.to_csv(csv_path, index=False)

print(f'Total baris diekspor: {len(df)} (harapan: 120)')
print('CSV disimpan ke:', csv_path)


Total baris diekspor: 120 (harapan: 120)
CSV disimpan ke: /content/drive/MyDrive/1_Metopen/Proposal Baru/v28_delivery/diagnostics_t17/t17_slope_30point_diagnostic.csv


/usr/local/lib/python3.13/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [7]:
# 6. Ringkasan Otomatis untuk Keputusan v28 (Tanpa Retrain)
summary = []
for route, grp in df.groupby('route'):
    valid = grp[grp['slope_valid'] == 1]
    lt5_pass = int((valid['slope_deg'] < 5).sum())
    gte5 = int((valid['slope_deg'] >= 5).sum())
    max_diff_hist = (valid['slope_deg'] - valid['historic_slope_srtm_training_export_deg']).abs().max()
    summary.append({
        'route': route,
        'n_rows': len(grp),
        'n_valid': len(valid),
        'n_slope_lt_5': lt5_pass,
        'n_slope_gte_5': gte5,
        'min_slope_deg': round(float(valid['slope_deg'].min()), 6),
        'max_slope_deg': round(float(valid['slope_deg'].max()), 6),
        'max_abs_diff_vs_historic_export': round(float(max_diff_hist), 8),
    })

summary_df = pd.DataFrame(summary)
summary_path = os.path.join(OUT_DIR, 't17_slope_route_summary.csv')
summary_df.to_csv(summary_path, index=False)

display(summary_df) if 'display' in globals() else print(summary_df.to_string(index=False))


                                    route  n_rows  n_valid  n_slope_lt_5  n_slope_gte_5  min_slope_deg  max_slope_deg  max_abs_diff_vs_historic_export
feature_slope_final_sampling_grid_control      30       30             0             30       5.103612      11.483837                         0.000000
         feature_srtm_slope_analysis_grid      30       30             0             30       5.103612      11.483837                         0.000000
          label_native_projection_control      30       30            30              0       1.316168       4.987200                        10.167669
 label_slope_forced_analysis_grid_control      30       30             8             22       4.986845      11.230177                         0.253660
